# 第九章：降维与生成模型——通向 GAN 的桥梁

本章连接两条线：t-SNE/UMAP 延续了 PCA 的降维主题，自编码器/VAE 则为下一章的 GAN 铺设生成模型基础。

## 9.1 t-SNE：看到 PCA 看不到的结构

PCA 是**线性**降维——它只能捕捉数据中的线性关系。当数据分布在高维空间中的
复杂曲面上时，PCA 会丢失重要结构。

**t-SNE (t-Distributed Stochastic Neighbor Embedding, 2008)** 是非线性降维的黄金标准。
它通过保持高维和低维空间中点对之间的**局部相似性**来揭示数据的内在聚类结构。

### PCA vs t-SNE

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **PCA** | 保留最大方差的线性方向 | 速度快，可解释 | 无法捕捉非线性结构 |
| **t-SNE** | 保持局部邻居相似性 | 揭示聚类，视觉漂亮 | 计算慢，随机性（不同运行结果不同） |


### 9.1.1 t-SNE 的数学原理

t-SNE 的核心思想：在高维空间中相似的样本点，在低维空间中也应该靠近。

#### Step 1: 高维相似性（高斯核）

对每个样本 $x_i$，定义它与其他样本的条件概率（"$x_j$ 是 $x_i$ 的邻居"）：

$$
p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}
$$

$\sigma_i$ 由 **perplexity** 参数控制，使得每个样本的有效邻居数大致等于 perplexity。

对称化：$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$

#### Step 2: 低维嵌入（t 分布）

在低维空间中，用自由度为 1 的 Student t 分布（即 Cauchy 分布）来度量相似性：

$$
q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}
$$

#### Step 3: KL 散度最小化

t-SNE 最小化 $P$ 和 $Q$ 之间的 KL 散度：
$$
\mathcal{L} = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

#### 为什么用 t 分布而非高斯？

高斯分布的尾部太薄——中距离的点对在低维空间中要么挤在一起（无法区分），要么推得太远。t 分布的**重尾 (heavy tail)** 属性允许中距离点对在低维空间中适度分开，产生更好的聚类分离效果。

### 9.1.2 t-SNE vs UMAP：非线性降维的选择困境

| 维度 | t-SNE | UMAP |
|------|-------|------|
| 数学基础 | 概率（KL 散度） | 拓扑（模糊单纯形） |
| 全局结构 | 较差（距离无意义） | 较好（保留部分全局关系） |
| 速度 | $O(n^2)$ | $O(n^{1.14})$ |
| 可复现性 | 随机，不同运行结果不同 | 使用随机种子可复现 |
| 最佳场景 | 聚类可视化 | 大规模数据探索 |


In [ ]:
# === PCA vs t-SNE 对比 ===
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import time

# 生成复杂数据
np.random.seed(42)
n_samples = 500

# 三维瑞士卷数据（Swiss Roll）
t = np.linspace(0, 15, n_samples)
X_swiss = np.zeros((n_samples, 3))
X_swiss[:, 0] = t * np.cos(t)
X_swiss[:, 1] = t * np.sin(t)
X_swiss[:, 2] = t
color = np.arctan2(X_swiss[:, 1], X_swiss[:, 0])

print("数据形状:", X_swiss.shape)

# PCA
t0 = time.time()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)
t_pca = time.time() - t0
print(f"PCA: {X_pca.shape}, 用时 {t_pca:.3f}s")

# t-SNE (需要 scikit-learn)
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_swiss)
t_tsne = time.time() - t0
print(f"t-SNE: {X_tsne.shape}, 用时 {t_tsne:.3f}s")

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
methods = [('PCA', X_pca, 'PCA降维'), 
           ('t-SNE', X_tsne, 't-SNE降维'),
           ('Original', X_swiss, '原始3D数据 (仅第一维度着色)')]

for i, (title, data, label) in enumerate(methods):
    ax = axes[i]
    if data.shape[1] == 3:  # 3D数据
        ax.scatter(data[:, 0], data[:, 1], data[:, 2],
                   c=color, s=3, alpha=0.5)
        ax.set_title(f'{title}\n({label})')
    else:
        ax.scatter(data[:, 0], data[:, 1], c=color, cmap='viridis', s=5)
        ax.set_title(f'{title}\n({label})')
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../fig/pca_vs_tsne.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n关键观察:")
print(f"  PCA: 找到 {X_pca.shape[1]} 维 → 丢失了非线性结构")
print(f"  t-SNE: 找到 {X_tsne.shape[1]} 维 → 发现了 {len(np.unique(color))} 个聚类")


## 9.2 UMAP：更快更稳定的替代方案

**UMAP (Uniform Manifold Approximation and Projection, 2018)** 是比 t-SNE 更新的
非线性降维方法。它在保持全局结构方面优于 t-SNE，同时运行速度更快、结果可复现。

### t-SNE 的痛点

- **随机性**：不同运行给出不同结果
- **perplexity**：关键超参数，控制局部 vs 全局结构的平衡（典型值 5-50）
- **计算成本**：$O(n^2)$ 或更高

### 选择建议

- 需要**可复现**的结果：使用 UMAP
- 需要**探索性分析**：t-SNE 多次运行观察聚类稳定性
- 需要**速度**：PCA 用于快速预览


In [ ]:
# === 手写数字 t-SNE 可视化 ===
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits 数据: {X_digits.shape}")

# PCA 快速预览
pca_digits = PCA(n_components=2).fit_transform(X_digits)
print(f"PCA (2D): 前两个主成分")

# t-SNE
tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne_d = tsne_digits.fit_transform(X_digits)
print(f"t-SNE (2D)")

# UMAP (如果可用)
try:
    import umap
    umap_reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = umap_reducer.fit_transform(X_digits)
    print(f"UMAP (2D): {X_umap.shape}")
except ImportError:
    print("UMAP 未安装。安装: pip install umap-learn")

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
plot_data = [
    ('PCA', X_pca),
    ('t-SNE', X_tsne_d),
]
try:
    plot_data.append(('UMAP', X_umap))
except NameError:
    pass

for i, (title, data) in enumerate(plot_data):
    ax = axes[i]
    scatter = ax.scatter(data[:, 0], data[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('../fig/tsne_umap_digits.png', dpi=100, bbox_inches='tight')
plt.show()


## 9.3 自编码器 (Autoencoder)：生成模型的基石

自编码器是最简单的**生成模型**——它学习将输入复制到输出。
虽然看起来"什么都没做"，但这个"复制"任务迫使网络学习
数据的**紧凑表示 (compact representation)**。

### 自编码器架构

```
输入 x → [Encoder] → 潜在码 z → [Decoder] → 重建 x̂
```

### 数学定义

- **Encoder**: $\mathbf{z} = f_\theta(\mathbf{x})$ 将输入压缩为低维潜在表示
- **Decoder**: $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$ 从潜在码重建输入
- **损失函数**: $\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$

### 为什么需要自编码器？

1. **降维**：学到数据的紧凑表示 → 可用于可视化、压缩、去噪
2. **生成**：从潜在空间采样 → 可生成新数据
3. **异常检测**：重建误差大的样本 → 可能是异常值


### 9.3.1 潜在空间与流形假设

自编码器的核心产物是**潜在空间 (Latent Space)**——将高维数据压缩到的低维连续空间。

#### 流形假设 (Manifold Hypothesis)

真实世界的高维数据（如图像、声音）虽然表面上是高维的（一张 28×28 图片 = 784 维），但实际上分布在低维流形上——所有合法的 MNIST 手写数字只占据 784 维空间中的一个极小区域。

自编码器做的事情就是**学习这个流形的参数化**：Encoder 将流形上的点映射到潜在空间坐标，Decoder 将坐标映射回流形上的点。

#### 为什么潜在空间是连续的？

一个训练良好的自编码器的潜在空间具有以下性质：
- **连续性**：潜在空间中相邻的点解码后是相似的图像
- **可插值**：在潜在空间中两个点之间线性插值，解码后得到平滑过渡的图像

这就是为什么 VAE 可以从潜在空间随机采样生成新图像——它学会了整个流形，而不仅仅是训练样本点。


In [ ]:
# === 简单自编码器 (MNIST) ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class Autoencoder(nn.Module):
    """简单全连接自编码器: 784 → 32 → 784"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 784),
            nn.Sigmoid()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat.view(x.size(0), 784)  # unflatten to 28x28

# 训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    './data', train=True,
    download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")
print("训练中...")

history = []
for epoch in range(10):
    model.train()
    total_loss = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)
        
        optimizer.zero_grad()
        x_hat, _ = model(x_batch)
        loss = criterion(x_hat, x_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    history.append(loss.item())
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}")

# 重建效果
model.eval()
with torch.no_grad():
    sample = next(iter(train_loader))[0].unsqueeze(0).to(device)
    recon = model(sample).view(28, 28)
    
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
# 原始
axes[0].imshow(train_dataset[0][0].view(28, 28), cmap='gray')
axes[0].set_title('Original')
# 重建
axes[1].imshow(recon.cpu().view(28, 28), cmap='gray')
axes[1].set_title(f'Reconstructed (epoch 10, loss={history[-1]:.4f})')
# 随机生成
z_random = torch.randn(1, 32).to(device)
generated = model.decoder(z_random.view(1, 32, 1, 1))
axes[2].imshow(generated.cpu().view(28, 28), cmap='gray')
axes[2].set_title('Generated from Random Latent Code')

plt.tight_layout()
plt.savefig('../fig/autoencoder.png', dpi=100, bbox_inches='tight')
plt.show()


## 9.4 变分自编码器 (VAE)：生成式模型的进化

VAE (Variational Autoencoder, 2013) 是自编码器的生成式扩展。它不满足于
"精确重建"，而是学习数据的**概率分布**——潜在空间中的每个点代表一个
概率分布，Decoder 输出的是**分布参数** $(\mu, \sigma)$。

### VAE 核心公式

$$
\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}
\left[\log p_\theta(\mathbf{x}|\mathbf{z})\right]}_{\text{重建误差}}
- \underbrace{D_{\text{KL}}\left(q_\phi(\mathbf{z}|\mathbf{x}) \|
p_\theta(\mathbf{z})\right)}_{\text{KL 正则化}}
$$

其中：
- **Encoder** $q_\phi(\mathbf{z}|\mathbf{x})$：从数据 $\mathbf{x}$ 推断潜在分布
- **Decoder** $p_\theta(\mathbf{x}|\mathbf{z})$：从潜在变量 $\mathbf{z}$ 生成数据
- 第一项是**重建损失**（希望 $\hat{\mathbf{x}} \approx \mathbf{x}$）
- 第二项是**KL 散度**（迫使近似后验 $q_\phi$ 接近先验 $p_\theta$）


In [ ]:
# === VAE 实现 (MNIST) ===
class VAE(nn.Module):
    """简单 VAE: 784 → (32, 32) → 2 → 784"""
    def __init__(self):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
        # Mu, Logvar → latent z
        self.fc_mu = nn.Linear(64, 2)
        self.fc_logvar = nn.Linear(64, 2)
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def decode(self, z):
        h = self.decoder(z)
        return torch.sigmoid(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat

model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    './data', train=True,
    download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

history = []
for epoch in range(10):
    model.train()
    total_loss = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)
        
        optimizer.zero_grad()
        mu, logvar = model.encode(x_batch)
        z = model.reparameterize(mu, logvar)
        x_hat = model.decode(z)
        
        loss_recon = F.mse_loss(x_hat, x_batch)
        loss_kl = F.kl_div(
            F.log_softmax(z, dim=1),
            F.log_softmax(model.reparameterize(mu, logvar), dim=1)
        )
        loss = loss_recon + 0.1 * loss_kl
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    history.append(loss.item())
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1}: recon_loss={loss_recon.item():.4f}, kl_loss={loss_kl.item():.4f}")

# 生成与重建
model.eval()
with torch.no_grad():
    z_random = torch.randn(1, 2).to(device)
    mu, logvar = model.encode(torch.randn(64, 1, 28, 28).to(device))
    sample = model.reparameterize(mu, logvar).view(1, 2, 28, 28)
    generated = model.decode(sample).view(1, 28, 28)

fig, axes = plt.subplots(2, 3, figsize=(14, 5))
axes[0, 0].imshow(train_dataset[0][0].view(28, 28), cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 1].imshow(generated.cpu().view(28, 28), cmap='gray')
axes[0, 1].set_title('Generated from VAE')
axes[1, 0].imshow(train_dataset[1][0].view(28, 28), cmap='gray')
axes[1, 0].set_title('Original (sample 1)')
axes[1, 1].imshow(generated.cpu().view(28, 28), cmap='gray')
axes[1, 1].set_title('Generated (sample 1)')

plt.tight_layout()
plt.savefig('../fig/vae_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n关键洞察:")
print("  PCA: 全局线性结构 → 10 个数字类别混在一起")
print("  t-SNE: 局部邻域结构 → 清晰的 10 个聚类")
print("  VAE: 连续潜在空间 → 10 个数字的平滑流形")


## 本章小结

| 方法 | 一句话总结 |
|------|-----------|
| **t-SNE** | PCA 看不到的非线性聚类结构 |
| **UMAP** | 比 t-SNE 更快更稳定的替代方案 |
| **Autoencoder** | 学习数据的紧凑表示，用于降维和生成 |
| **VAE** | Autoencoder 的生成式升级版，学习概率分布 |

✅ 非线性降维、生成模型基础完成。现在你有了从经典 ML 到生成式 AI 的完整武器库。
